# Text Summarization API — Development Notebook

This notebook demonstrates how to:
1. Load the `facebook/bart-large-cnn` model directly from Hugging Face
2. Run summarization locally to explore model behavior
3. Call the FastAPI server via `requests`

**Model page:** https://huggingface.co/facebook/bart-large-cnn

## 1. Install dependencies

In [ ]:
# Run this cell only if packages are not yet installed
!pip install transformers torch fastapi uvicorn requests sentencepiece -q

## 2. Load the model locally and test inference

> Uses `AutoTokenizer` + `AutoModelForSeq2SeqLM` directly (compatible with transformers v5+).

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
print("Model loaded!")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:03<00:00, 133.82it/s]


Model loaded!


In [4]:
def summarize(text: str, max_length: int = 130, min_length: int = 30) -> str:
    """Tokenize, generate, and decode a summary for the given text."""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True,
    )
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True,
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [5]:
TEXT_1 = """
Artificial intelligence has rapidly transformed numerous industries over the past decade.
From healthcare to finance, AI-powered tools are streamlining processes, reducing costs,
and enabling capabilities that were previously impossible. Machine learning models can now
diagnose diseases from medical images with accuracy rivaling human specialists.
In the financial sector, algorithmic trading systems process millions of transactions per
second, while fraud detection models protect consumers in real time.
Natural language processing has made virtual assistants and chatbots ubiquitous,
handling customer service queries around the clock without human intervention.
"""

summary1 = summarize(TEXT_1, max_length=80, min_length=20)
print("Summary:", summary1)

Summary: From healthcare to finance, AI-powered tools are streamlining processes. Machine learning models can nowdiagnose diseases from medical images with accuracy rivaling human specialists.


In [6]:
TEXT_2 = """
Scientists have discovered a new exoplanet located approximately 40 light-years from Earth
that may have conditions suitable for liquid water on its surface.
The planet, designated K2-18b, orbits within the habitable zone of its host star,
a red dwarf known as K2-18. Using data from the James Webb Space Telescope,
researchers detected the chemical signatures of carbon dioxide and methane in the planet's
atmosphere — molecules that on Earth are associated with biological activity.
While scientists caution that these findings are not conclusive evidence of life,
they represent one of the most promising leads in the ongoing search for habitable worlds
beyond our solar system.
"""

summary2 = summarize(TEXT_2, max_length=100, min_length=30)
print("Summary:", summary2)

Summary: The planet, designated K2-18b, orbits within the habitable zone of its host star. Using data from the James Webb Space Telescope, scientists detected the chemical signatures of carbon dioxide and methane in the planet's atmosphere.


## 3. Call the FastAPI server via `requests`

> Make sure the server is running:  
> ```bash
> uvicorn main:app --reload
> ```

In [7]:
import requests, json

BASE_URL = "http://127.0.0.1:8000"

# Health check
resp = requests.get(f"{BASE_URL}/health")
print("Health:", resp.json())

Health: {'status': 'ok', 'model_loaded': True, 'model': 'facebook/bart-large-cnn'}


In [8]:
# POST /predict
payload = {
    "text": TEXT_1,
    "max_length": 80,
    "min_length": 20,
}
resp = requests.post(f"{BASE_URL}/predict", json=payload)
print(json.dumps(resp.json(), indent=2))

{
  "model": "facebook/bart-large-cnn",
  "summary": "Artificial intelligence has rapidly transformed numerous industries over the past decade. From healthcare to finance, AI-powered tools are streamlining processes, reducing costs, and enabling capabilities that were previously impossible.",
  "input_word_count": 86,
  "summary_word_count": 29,
  "processing_time_seconds": 5.209
}


In [9]:
# Error case: empty text
resp_err = requests.post(f"{BASE_URL}/predict", json={"text": ""})
print("Status:", resp_err.status_code)
print(json.dumps(resp_err.json(), indent=2))

Status: 422
{
  "detail": [
    {
      "type": "value_error",
      "loc": [
        "body",
        "text"
      ],
      "msg": "Value error, 'text' must not be empty.",
      "input": "",
      "ctx": {
        "error": {}
      }
    }
  ]
}
